<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 1 — Python, NumPy, pandas & Matplotlib Refresher</div>
</div>

#### This notebook revises the Python skills introduced in PHYS281 that the rest of this module builds on.
***

## <span style="color:#B5121B">How to use this notebook</span>
This notebook revises material from PHYS281. Run every cell, change the values and see what breaks. There are short exercises after most sections: do them as you go.
***

## NumPy revision

In [ ]:
import numpy as np

# Arrays behave like vectors: operations apply elementwise, without an explicit loop
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y = np.array([10.0, 20.0, 30.0, 40.0, 50.0])
print("x + y      =", x + y)
print("x * 2      =", x * 2)
print("mean(x)    =", x.mean())
print("std(x)     =", x.std())

**Vectorisation matters.** A NumPy array operation runs as compiled C code rather than as a Python loop. Compare the two for the same calculation:

In [ ]:
import timeit

n = 2_000_000
data = np.random.default_rng(0).uniform(0, 1, size=n)

def loop_version():
    total = 0.0
    for v in data:
        total += v ** 2
    return total

def vectorised_version():
    return (data ** 2).sum()

# Take the best of a few repeats - a single timing is noisy, especially
# the very first call, which pays a one-off warm-up cost unrelated to the
# calculation itself.
t_loop = min(timeit.repeat(loop_version, number=1, repeat=3))
t_vec = min(timeit.repeat(vectorised_version, number=1, repeat=3))

print(f"Python loop : {t_loop:.4f} s   (result={loop_version():.4f})")
print(f"Vectorised  : {t_vec:.5f} s   (result={vectorised_version():.4f})")
print(f"Speed-up    : {t_loop / t_vec:.0f}x")

### From loops to linear algebra

`(data ** 2).sum()` isn't just a faster loop: it describes the calculation in a different way. Compare a dot product computed both ways:

In [ ]:
a = np.array([1.0, 2.0, 3.0, 4.0])
b = np.array([5.0, 6.0, 7.0, 8.0])

# As a loop - exactly what you'd write in plain Python
total = 0.0
for i in range(len(a)):
    total += a[i] * b[i]
print("loop result :", total)

# As linear algebra: a dot product
print("np.dot      :", np.dot(a, b))
print("@ operator  :", a @ b)

The loop `for i in range(len(a)): total += a[i] * b[i]`, the expression $\mathbf{a}\cdot\mathbf{b} = \sum_i a_i b_i$ and `a @ b` are the *same calculation*. `@` (and `np.dot`) performs the loop above in compiled code.

The same holds for matrices. A matrix-vector multiplication is a loop of dot products, one per row:

In [ ]:
# A 3x4 matrix (e.g. three different weighted combinations of four inputs)
M = np.array([
    [1.0, 0.0, 2.0, 1.0],
    [0.0, 1.0, 0.0, 3.0],
    [2.0, 1.0, 1.0, 0.0],
])
v = np.array([1.0, 2.0, 3.0, 4.0])

# As nested loops
result_loop = np.zeros(M.shape[0])
for i in range(M.shape[0]):          # one dot product per row...
    row_total = 0.0
    for j in range(M.shape[1]):      # ...computed as a loop over columns
        row_total += M[i, j] * v[j]
    result_loop[i] = row_total
print("loop result   :", result_loop)

# As linear algebra
print("matrix result :", M @ v)

Two nested loops (rows, then columns) *are* a matrix-vector multiplication. Replacing a loop with an array operation means rewriting the calculation as linear algebra.

> **Quick check:** without running it, what shape would `result_loop` be if `M` were 5×4 instead of 3×4? How does `v @ M.T` compare with `M @ v`: same numbers, a different shape, or something else? Run it to check your answer.

***
## pandas: loading and exploring data

`pandas` is built on NumPy and handles labelled, tabular data: rows and columns, like a spreadsheet, but scriptable.

In [ ]:
import pandas as pd
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week1/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

df = pd.read_csv(data_path("pandas_practice_dataset.csv"))
df.head()

### Selecting, slicing and masking

In [ ]:
# A boolean mask selects rows matching a condition
high_calorie = df[df["Calories"] > 400]
high_calorie

In [ ]:
# Selecting specific columns
df[["Date", "Pulse", "Calories"]].head()

### Grouping and aggregating

> **If you've used SQL:** `.groupby()` does the same job as `GROUP BY`, the boolean mask above (`df[df["Calories"] > 400]`) is `WHERE`, and `.merge()` is `JOIN`. The syntax differs, but the operations (group, filter, combine tables) are the same as in a database query.

In [ ]:
# Mean calories burned, grouped by session duration
df.groupby("Duration")["Calories"].mean()

> **Quick check:** the `Maxpulse` column exists alongside `Pulse`. Compute, in one line, the mean *difference* between them (`Maxpulse - Pulse`) for each `Duration` group. Is the gap bigger for longer or shorter sessions?

***
## Matplotlib basics

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["Pulse"], df["Calories"], "o", color="dodgerblue")
ax.set_xlabel("Pulse (bpm)")
ax.set_ylabel("Calories burned")
ax.set_title("Calories vs. pulse")
plt.show()

In [ ]:
# A histogram, and an errorbar plot with synthetic uncertainties
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df["Calories"], bins=8, color="mediumseagreen", edgecolor="black")
axes[0].set_xlabel("Calories burned")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of calories burned")

rng = np.random.default_rng(1)
y_err = rng.uniform(5, 20, size=len(df))
axes[1].errorbar(df["Duration"], df["Calories"], yerr=y_err, fmt="o", color="firebrick", capsize=3)
axes[1].set_xlabel("Duration (min)")
axes[1].set_ylabel("Calories burned")
axes[1].set_title("With (illustrative) uncertainties")

fig.tight_layout()
plt.show()

### Saving figures

Save figures at a sensible resolution, with a tight bounding box, and in a vector format (PDF) where possible, so they stay sharp at any zoom.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["Pulse"], df["Calories"], "o", color="dodgerblue")
ax.set_xlabel("Pulse (bpm)")
ax.set_ylabel("Calories burned")
fig.savefig("pulse_vs_calories.pdf", bbox_inches="tight", dpi=200)
print("Saved.")

***
## <span style="color:#FD9029">Exercises</span>

These become less prescriptive as you go; the last two are open-ended.

1. Compute the mean **and** standard deviation of `Calories` for each `Duration` group (hint: `.agg(["mean", "std"])`). Make a single, clearly labelled bar chart of the mean calories per group, with the standard deviation as error bars. Save it as a PDF.
2. Rewrite the matrix-vector multiplication from the NumPy section so that it computes `M @ v` using only `np.dot` in a loop over rows (no `@`, no nested loop). This sits between the fully looped and the fully vectorised versions.
3. The `Date` column is stored as text. Convert it to a datetime type with `pd.to_datetime`, then plot `Calories` against date as a line, with real dates on the x-axis.
   *Hint: look at the raw `Date` values first. Some are wrapped in quote marks, one uses a different format and one is missing, so `pd.to_datetime` will fail until the column is tidied. `.str.strip()` and the `format="mixed"` and `errors="coerce"` options will help.*
4. **Open-ended:** using only the tools in this notebook, find and plot one relationship in the data not shown above (e.g. does `Maxpulse` predict `Calories` better than `Pulse`?). Produce one figure and one sentence stating what you found.
***

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>